# 11 全コア同時GDとDMRG-like（2コア局所更新）の比較

この Notebook が扱う問いは次である。

> 同じ初期TTモデルと全学習データの平均cross-entropy勾配を用いる条件で、全コア同時GDと、DMRG-like（2コア局所更新）は、性能と学習費用にどのような違いを生むか。

これは、2コア局所更新だけの因果効果を取り出す比較ではない。結論は、全コア同時GDと、DMRG-like（2コア局所更新）全体の比較に限る。

同じ full-dataset gradient evaluation 数を、同じ計算費用とは呼ばない。全コア同時GDの1回は全コアと通常パラメータを更新し、DMRG-like（2コア局所更新）の1回は対象の2コアと通常パラメータを更新する。1回当たりの演算量、メモリ、実時間は異なる。


## 共通条件と、流用する既存実験

土台は Notebook 06 の Fashion-MNIST TT-MLP 比較である。GPU 常駐の full-batch 供給と更新対象の監査は Notebook 05、時間と peak memory の測り方は Notebook 04、学習率の候補と条件の持ち方は Notebook 02 と 03、保存の世代管理は Notebook 07 の `publish_generation` / `load_active_generation` を使う。

- Fashion-MNIST。既存の 60,000 件を、学習 50,000、early-stopping validation 5,000、rank-selection validation 5,000 に分ける。分割 seed は 0。
- test は取得した直後に捨て、件数も中身も見ない。
- 6-core `TTLinear`。`out_modes = (4, 2, 2, 2, 4, 4)`、`in_modes = (2, 2, 2, 2, 7, 7)`。
- rank 上限 12。初期の bond 次元は `(1, 8, 12, 12, 12, 12, 1)`。
- 初期状態は Notebook 01 の Stage C checkpoint。site 0 を中心とする正準形で、両方式へ同じ core 値を渡す。
- dtype は float32。学習を実行するときの device は Notebook 06 と同じ CUDA。
- 損失は平均 cross-entropy。初期状態も best 候補に含める。
- 学習率は事前に `(0.1, 0.2, 0.3)` だけ。0.3 は Notebook 05 と 06 の joint 学習率、0.2 は Notebook 03 の joint 学習率、0.1 は Notebook 02 の最大倍率 100 が作る joint 学習率 `0.001 * 100` である。TT 側と通常 parameter 側で別の学習率は使わない。結果を見て候補を足さない。
- 選択に使うのは early-stopping validation だけである。rank-selection validation は最終比較に使えるが、Notebook 06 までで参照済みの開発用データであり、独立した最終評価ではない。

## 二つの更新

全コア同時GD：すべてのTTコアと通常パラメータの勾配を求め、まとめてGDで更新する方式。

DMRG-like（2コア局所更新）：隣接2コアを結合してGDで更新し、SVDで2コアへ戻しながら左右に巡回する方式。通常パラメータも各局所更新で更新する。物理DMRGの局所固有値問題を解く方式ではない。

全コア同時GDでは、全 TT core、`fc1.bias`、`fc2`、`fc3` を、学習 50,000 件の平均 cross-entropy 勾配で同時に更新する。momentum、Adam、weight decay は使わない。core shape は変えない。rank を維持するための SVD や再正準化は足さない。実効 weight の最小 TT-rank が一定だとは扱わない。記録する bond 次元は core shape から読んだ値である。実装は `update_tt_linear_global_full_batch` である。

DMRG-like（2コア局所更新）の1往復は、既存の `train_scoped_full_batch_budget` が行う joint 更新である。`tt_linear_scoped_update.py` は pair 更新を含むため、全コア同時GDの実装としては呼ばない。

## 評価間隔と停止

core 数を $d$、各 pair の局所更新回数を `LOCAL_STEPS_PER_PAIR` とする。1 往復の full-dataset gradient evaluation 数は

$$
2(d-1)\times\mathrm{LOCAL\_STEPS\_PER\_PAIR}
$$

である。既存の $d = 6$、`LOCAL_STEPS_PER_PAIR = 8` では 80 になるが、コードはこの式から計算する。全コア同時GDも同じ回数ごとに評価する。DMRG-like（2コア局所更新）は 1 往復の終了時、全コア同時GDは対応するデータ利用回数の終了時である。初期状態も両方式で評価する。

停止は両方式とも `update_validation_selection` に従う。`min_delta = 10^{-4}`、patience は 5、評価間隔は上の式、最大 full-dataset gradient evaluation 数は Notebook 06 の 200 往復に対応する 16,000 で、評価間隔の整数倍である。best は strict best であり、同値の loss は改善にしない。非有限値は `NonFiniteLocalUpdate` のまま止め、別の停止理由へ読み替えない。停止理由は `early_stopping`、`max_gradient_evaluations`、`non_finite`、`out_of_memory` のいずれかであり、`converged` は使わない。

学習率は、`early_stopping` または `max_gradient_evaluations` で終わった条件だけから、early-stopping validation の strict best loss が最小の条件を選ぶ。`non_finite` と `out_of_memory` は最良条件にしない。ある方式の全学習率が失敗した場合は「選択可能条件なし」とし、その方式の rank-selection 評価は行わない。loss が同値なら strict best の gradient evaluation 数が小さい方、それも同値なら累積学習秒が小さい方を選ぶ。最終比較は、こうして選んだ各方式の best checkpoint である。学習率探索に使った全条件の時間も保存する。

## 既存成果物

再利用は Notebook の hash だけでは判定しない。履歴は更新数 0 から始まり、評価間隔の欠落・重複・逆行がなく、update log の完了更新数と一致することを見る。学習時の実装と対応づけられない項目は unknown とし、完全一致確認済みにはしない。初期 checkpoint、分割、tensorization、rank、更新対象、全データ勾配の定義、学習率、pair 巡回、`LOCAL_STEPS_PER_PAIR`、評価間隔、validation selection、`min_delta`、patience、最大予算、best と final の定義が一致し、保存済み履歴をその選択規則で再生できるときだけ再集計する。条件が違う保存は主比較へ混ぜず、監査表に参考として残す。既存ファイルは削除も上書きもしない。Notebook 06 の DMRG-like は、保存時の学習実装と現行実装の同一性を確認できないため参考値に限り、主比較へは混ぜない。主比較用の DMRG-like（2コア局所更新）、学習率 0.3 は新規実行する。`TRAINING_CONTRACT_VERSION` は、全コア同時GDとDMRG-like（2コア局所更新）の学習挙動、停止、保存内容を変えたときだけ増やす。図、キャプション、表示用 DataFrame、描画関数だけの変更では変えない。学習 condition の run ID は、学習設定、データ checksum、Stage C checkpoint、学習に使う src 実装 hash、この version から作る。Notebook 全コードの hash は run ID に入れず、記録用の `notebook_code_sha256` として config に残す。

実行スイッチは保存時点で次のとおりすべて偽である。

```python
RUN_SMOKE = False
RUN_MAIN_EXPERIMENT = False
REGENERATE_DISPLAY_ARTIFACTS = False
```


In [17]:
from __future__ import annotations

import copy
import csv
import shutil
import hashlib
import json
import math
import time
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display
from matplotlib import font_manager
from torch import nn
from torch.utils.data import DataLoader

from nn_compression.compression import TTLinear
from nn_compression.datasets import get_fashion_mnist_datasets, split_fashion_mnist_dataset
from nn_compression.metrics.evaluation import evaluate
from nn_compression.models import MNISTMLP
from nn_compression.training import (
    GpuResidentTensorBatches,
    NonFiniteLocalUpdate,
    TTUpdateScope,
    initial_validation_selection,
    load_tt_linear_sweep_checkpoint,
    train_scoped_full_batch_budget,
    update_tt_linear_global_full_batch,
    update_tt_linear_two_core_pair,
    update_validation_selection,
)
from nn_compression.tt_mps import tt_matrix_canonicality_errors, tt_matrix_ranks
from nn_compression.utils import (
    atomic_torch_save,
    atomic_write_csv,
    atomic_write_json,
    file_sha256,
    find_project_root,
    index_sha256,
    load_active_generation,
    notebook_code_sha256,
    publish_generation,
    tensor_sha256,
)

RUN_SMOKE = False
RUN_MAIN_EXPERIMENT = False
REGENERATE_DISPLAY_ARTIFACTS = True

SEED = 0
SPLIT_LENGTHS = (50_000, 5_000, 5_000)
OUT_MODES = (4, 2, 2, 2, 4, 4)
IN_MODES = (2, 2, 2, 2, 7, 7)
RANK_CAP = 12
EXPECTED_ACTUAL_RANKS = (1, 8, 12, 12, 12, 12, 1)
N_CORES = len(OUT_MODES)
LOCAL_STEPS_PER_PAIR = 8
EVALUATION_INTERVAL = 2 * (N_CORES - 1) * LOCAL_STEPS_PER_PAIR
MAX_SWEEPS = 200
MAX_GRADIENT_EVALUATIONS = MAX_SWEEPS * EVALUATION_INTERVAL
MIN_DELTA = 1e-4
PATIENCE = 5
LEARNING_RATES = (0.1, 0.2, 0.3)
GRADIENT_BATCH_SIZE = 4096
VALIDATION_BATCH_SIZE = 512
DIAGNOSTIC_BATCH_SIZE = 512
TARGET_LAYER = "fc1"
GLOBAL_METHOD = "global_gd"
LOCAL_METHOD = "local_sweep"
METHODS = (GLOBAL_METHOD, LOCAL_METHOD)
FINISHED_STOPS = {"early_stopping", "max_gradient_evaluations", "non_finite", "out_of_memory"}
COMPARISON_STOPS = {"early_stopping", "max_gradient_evaluations"}
# Global/Local の学習挙動、停止、保存内容を変えたときだけ増やす。
# 図、キャプション、表示用 DataFrame、描画関数だけの変更では増やさない。
TRAINING_CONTRACT_VERSION = 1

BEST_RECHECK_LOSS_ATOL = 1e-5
BEST_RECHECK_ACCURACY_ATOL = 1e-6
IMPLEMENTATION_FILES = (
    "src/nn_compression/training/tt_linear_global_full_batch.py",
    "src/nn_compression/training/tt_linear_scoped_update.py",
    "src/nn_compression/training/tt_linear_full_batch_sweep.py",
    "src/nn_compression/training/validation_selection.py",
    "src/nn_compression/utils/experiment_generation.py",
)
NOTEBOOK06_IMPLEMENTATION_FILES = (
    "src/nn_compression/training/validation_selection.py",
    "src/nn_compression/training/tt_linear_scoped_update.py",
    "src/nn_compression/training/loops.py",
    "src/nn_compression/metrics/macs.py",
    "src/nn_compression/metrics/inference_latency.py",
)
TIMING_SCOPE = (
    "cumulative_train_seconds は、完了した評価ブロックの parameter 更新秒だけを積む。"
    "性能対学習時間の図は、この秒を使う。"
    "cumulative_eval_seconds は、完了ブロックの validation 評価秒。"
    "Local は train_scoped_full_batch_budget が各往復の開始時と終了時を評価するため、"
    "ブロック境界の評価秒が隣の呼び出しと重なる。"
    "勾配評価回数は境界を二重に数えない。"
    "wall-clock time、total evaluation time、end-to-end の総費用には、この重複した Local 評価を含む。"
    "失敗してロールバックした試行の秒は failed_attempt に残し、完了履歴へは足さない。"
    "保存の overhead は学習時間に含めない。"
    "同じ full-dataset gradient evaluation 数は同じ計算費用ではない。"
)
PEAK_MEMORY_SCOPE = (
    "新規実行は、条件開始時に reset_peak_memory_stats したあと、"
    "完了ブロックの更新、validation 評価、best の保持、世代保存までに測った max_memory_allocated。"
    "status が recorded の値だけを、この範囲の peak として図に載せる。"
    "Notebook 06 の peak_training_allocated_bytes は当時の記録値であり、測定範囲は未検証。"
    "不明な値は 0 に置き換えない。"
)
SMOKE_LOCAL_STEPS_PER_PAIR = 1
SMOKE_EVALUATION_INTERVAL = 2 * (N_CORES - 1) * SMOKE_LOCAL_STEPS_PER_PAIR
SMOKE_MAX_GRADIENT_EVALUATIONS = SMOKE_EVALUATION_INTERVAL
SMOKE_LEARNING_RATES = (0.3,)
STAGE_C_SHA256 = "ef9d5afcbffc0cdad787bc7e17f230917ae7cfb780bd506f5a348ae38ce10c6a"
SPLIT_INDEX_SHA256 = (
    "aaed0a23690e2e3c0fd3b21c21ec28adb2606b17941b98087503382c3372f913"
    "|12cf7fe6cc2cdbb43177107c8a30dffb86cc099386d61fa5f4f05fdb1795d360"
    "|a80ccd6c95a3aa08fcfb7a5a10ffc883344da48ec37ce605220f964cef14cb26"
)
IMAGE_SHA256 = "6a1fd0602db5a1066210a55375d18f9b12115269e7c6759f64504735f020d4d5"
LABEL_SHA256 = "6ead6f3f16028865d36fd3c8625eba751820bd8031265a3be03e206136a3c96c"
EXPECTED_DENSE_PARAMETERS = (
    "fc1.bias",
    "fc2.weight",
    "fc2.bias",
    "fc3.weight",
    "fc3.bias",
)
EXPERIMENT_NAME = "11_full_batch_global_vs_local_update_control"
SMOKE_EXPERIMENT_NAME = EXPERIMENT_NAME + "_smoke"
NOTEBOOK06_NAME = "06_dense_tt_dmrg_like_convergence_comparison"
PROTECTED_NAMES = {
    "00_full_batch_two_core_sweep_training",
    "01_tt_initialization_scale_diagnostics",
    "02_full_batch_sweep_update_budget_grid",
    "02_full_batch_sweep_update_budget_grid_smoke",
    "02_full_batch_sweep_update_budget_grid_smoke_bs512",
    "03_joint_learning_rate_02_03_s40_l8",
    "03_joint_learning_rate_02_03_s40_l8_smoke",
    "04_full_batch_gradient_optimization_benchmark",
    "05_update_scope_ablation_gpu_resident",
    "05_update_scope_ablation_gpu_resident_smoke",
    "06_dense_tt_dmrg_like_convergence_comparison",
    "06_dense_tt_dmrg_like_convergence_comparison_smoke",
    "07_fixed_rank_sweep_and_pareto",
    "07_fixed_rank_sweep_and_pareto_gpu_slice",
    "08_adaptive_bond_rank_dmrg_like_pareto",
    "09_posthoc_single_bond_truncation_diagnostic",
    "10_nonuniform_bond_rank_allocation_criteria",
}

if not isinstance(RUN_SMOKE, bool) or not isinstance(RUN_MAIN_EXPERIMENT, bool) or not isinstance(REGENERATE_DISPLAY_ARTIFACTS, bool):
    raise RuntimeError("実行スイッチは bool である必要があります。")
if EVALUATION_INTERVAL != 2 * (len(OUT_MODES) - 1) * LOCAL_STEPS_PER_PAIR:
    raise RuntimeError("評価間隔が 2(d-1) * LOCAL_STEPS_PER_PAIR と一致しません。")
if MAX_GRADIENT_EVALUATIONS % EVALUATION_INTERVAL != 0 or SMOKE_MAX_GRADIENT_EVALUATIONS % SMOKE_EVALUATION_INTERVAL != 0:
    raise RuntimeError("最大勾配評価数は評価間隔の整数倍である必要があります。")
if LEARNING_RATES != (0.1, 0.2, 0.3):
    raise RuntimeError("学習率候補が事前固定の (0.1, 0.2, 0.3) から変わっています。")

PROJECT_ROOT = find_project_root(Path.cwd())
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "40_dmrg" / "10_fashion_mnist_mlp" / f"{EXPERIMENT_NAME}.ipynb"
INITIAL_CHECKPOINT = PROJECT_ROOT / "models" / "40_dmrg" / "10_fashion_mnist_mlp" / "01_tt_initialization_scale_diagnostics" / "initial_stage_c.pt"
NOTEBOOK06_DMRG = PROJECT_ROOT / "models" / "40_dmrg" / "10_fashion_mnist_mlp" / NOTEBOOK06_NAME / "dmrg_like"


## 保存先と世代

学習の checkpoint は `models/.../11_full_batch_global_vs_local_update_control/` の条件ディレクトリへ、`publish_generation` で世代として書く。集計表と図は同じ実験名の `results/` へ書く。smoke は `_smoke` の兄弟ディレクトリで、main の state としては読まない。00 から 10 の保存済み成果物は読めるが、この Notebook からは書かない。


In [18]:
HISTORY_FIELDS = (
    "method", "learning_rate", "evaluation_block", "full_dataset_gradient_evaluations",
    "local_updates", "local_updates_status", "global_updates", "global_updates_status",
    "processed_samples", "train_loss", "train_accuracy", "validation_loss", "validation_accuracy",
    "core_gradient_norm", "core_gradient_norm_status", "dense_gradient_norm", "dense_gradient_norm_status",
    "joint_gradient_norm", "joint_gradient_norm_status", "core_relative_update", "core_relative_update_status",
    "dense_relative_update", "dense_relative_update_status", "joint_relative_update", "joint_relative_update_status",
    "bond_ranks", "cumulative_train_seconds", "cumulative_eval_seconds", "wall_clock_seconds",
    "is_strict_best", "strict_best_loss", "strict_best_step", "reference_loss",
    "non_improvement_count", "early_stop_triggered", "condition_stop_reason",
)
UPDATE_FIELDS = (
    "method", "learning_rate", "full_dataset_gradient_evaluations", "pair_index", "direction",
    "local_step", "center_before", "center_after", "retained_rank", "discarded_squared_norm",
    "joint_gradient_norm", "core_gradient_norm", "dense_gradient_norm", "joint_relative_update",
    "core_relative_update", "dense_relative_update", "full_batch_loss", "update_seconds", "finite",
)
INT_FIELDS = {
    "evaluation_block", "full_dataset_gradient_evaluations", "local_updates", "global_updates",
    "processed_samples", "strict_best_step", "non_improvement_count", "pair_index", "local_step",
    "center_before", "center_after", "retained_rank",
}
FLOAT_FIELDS = {
    "learning_rate", "train_loss", "train_accuracy", "validation_loss", "validation_accuracy",
    "core_gradient_norm", "dense_gradient_norm", "joint_gradient_norm", "core_relative_update",
    "dense_relative_update", "joint_relative_update", "cumulative_train_seconds",
    "cumulative_eval_seconds", "wall_clock_seconds", "strict_best_loss", "reference_loss",
    "discarded_squared_norm", "full_batch_loss", "update_seconds",
}


def experiment_dirs(smoke):
    name = SMOKE_EXPERIMENT_NAME if smoke else EXPERIMENT_NAME
    return (
        PROJECT_ROOT / "models" / "40_dmrg" / "10_fashion_mnist_mlp" / name,
        PROJECT_ROOT / "results" / "40_dmrg" / "10_fashion_mnist_mlp" / name,
    )


def assert_writable(path):
    resolved = Path(path).resolve()
    if set(resolved.parts) & PROTECTED_NAMES:
        raise RuntimeError(f"既存実験の保存先へは書きません: {resolved}")
    allowed = []
    for smoke in (False, True):
        allowed.extend(experiment_dirs(smoke))
    if not any(resolved == root.resolve() or root.resolve() in resolved.parents for root in allowed):
        raise RuntimeError(f"Notebook 11 のディレクトリ以外へは書きません: {resolved}")
    return resolved


def blank(value, status):
    if status != "recorded":
        return None, status
    return value, "recorded"


def mean_or_missing(values):
    finite = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    if len(finite) != len(values) or not finite:
        return None, "not_recorded"
    return sum(finite) / len(finite), "recorded"


def wall_clock_offset(history):
    """新規条件は 0、resume は保存済み最終行の wall-clock 秒。"""
    if not history:
        return 0.0
    return float(history[-1]["wall_clock_seconds"])


def resumed_wall_clock_seconds(offset, started_at, now):
    return float(offset) + (float(now) - float(started_at))


def require_monotonic_wall_clock(history):
    previous = None
    for row in history:
        current = float(row["wall_clock_seconds"])
        if previous is not None and current < previous:
            raise RuntimeError("wall-clock 時間が減少しています。")
        previous = current


def history_row(method, learning_rate, block, evaluations, selection, metrics, clocks, bond_ranks, counts):
    local_updates, local_status = counts["local"]
    global_updates, global_status = counts["global"]
    row = {
        "method": method,
        "learning_rate": learning_rate,
        "evaluation_block": block,
        "full_dataset_gradient_evaluations": evaluations,
        "local_updates": local_updates,
        "local_updates_status": local_status,
        "global_updates": global_updates,
        "global_updates_status": global_status,
        "processed_samples": evaluations * SPLIT_LENGTHS[0],
        "train_loss": metrics["train_loss"],
        "train_accuracy": metrics["train_accuracy"],
        "validation_loss": metrics["validation_loss"],
        "validation_accuracy": metrics["validation_accuracy"],
        "bond_ranks": "|".join(str(rank) for rank in bond_ranks),
        "cumulative_train_seconds": clocks["train"],
        "cumulative_eval_seconds": clocks["eval"],
        "wall_clock_seconds": clocks["wall"],
        "is_strict_best": bool(selection.strict_best_step == evaluations),
        "strict_best_loss": selection.strict_best_loss,
        "strict_best_step": selection.strict_best_step,
        "reference_loss": selection.reference_loss,
        "non_improvement_count": selection.non_improvement_count,
        "early_stop_triggered": bool(selection.early_stop_triggered),
    }
    for name in (
        "core_gradient_norm", "dense_gradient_norm", "joint_gradient_norm",
        "core_relative_update", "dense_relative_update", "joint_relative_update",
    ):
        value, status = metrics[name]
        row[name] = value
        row[name + "_status"] = status
    return row


def missing_metrics():
    missing = {}
    for name in (
        "core_gradient_norm", "dense_gradient_norm", "joint_gradient_norm",
        "core_relative_update", "dense_relative_update", "joint_relative_update",
    ):
        missing[name] = (None, "not_applicable")
    return missing


def read_table(path):
    with Path(path).open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))


def coerce_row(row):
    restored = {}
    for key, value in row.items():
        if value is None or value == "":
            restored[key] = None
        elif key in INT_FIELDS:
            restored[key] = int(float(value))
        elif key in FLOAT_FIELDS:
            restored[key] = float(value)
        elif value in {"True", "False"}:
            restored[key] = value == "True"
        else:
            restored[key] = value
    return restored


def selection_from_history(history):
    first = history[0]
    state = initial_validation_selection(
        float(first["validation_loss"]), int(first["full_dataset_gradient_evaluations"])
    )
    if int(first["full_dataset_gradient_evaluations"]) != 0:
        raise RuntimeError("履歴の先頭が初期状態ではありません。")
    for row in history[1:]:
        state = update_validation_selection(
            state,
            float(row["validation_loss"]),
            int(row["full_dataset_gradient_evaluations"]),
            min_delta=MIN_DELTA,
            patience=PATIENCE,
        )
        if state.early_stop_triggered:
            break
    return state



def combined_file_sha256(relative_paths):
    digest = hashlib.sha256()
    for relative in relative_paths:
        digest.update((PROJECT_ROOT / relative).read_bytes())
    return digest.hexdigest()


def implementation_sha256():
    return combined_file_sha256(IMPLEMENTATION_FILES)


def notebook06_implementation_sha256():
    return combined_file_sha256(NOTEBOOK06_IMPLEMENTATION_FILES)


def canonical_settings(smoke):
    rates = SMOKE_LEARNING_RATES if smoke else LEARNING_RATES
    interval = SMOKE_EVALUATION_INTERVAL if smoke else EVALUATION_INTERVAL
    maximum = SMOKE_MAX_GRADIENT_EVALUATIONS if smoke else MAX_GRADIENT_EVALUATIONS
    local_steps = SMOKE_LOCAL_STEPS_PER_PAIR if smoke else LOCAL_STEPS_PER_PAIR
    return {
        "experiment": SMOKE_EXPERIMENT_NAME if smoke else EXPERIMENT_NAME,
        "seed": SEED,
        "dtype": "float32",
        "device_policy": "cuda",
        "split_lengths": list(SPLIT_LENGTHS),
        "split_index_sha256": SPLIT_INDEX_SHA256,
        "image_sha256": IMAGE_SHA256,
        "label_sha256": LABEL_SHA256,
        "initial_checkpoint_sha256": STAGE_C_SHA256,
        "out_modes": list(OUT_MODES),
        "in_modes": list(IN_MODES),
        "rank_cap": RANK_CAP,
        "expected_actual_ranks": list(EXPECTED_ACTUAL_RANKS),
        "learning_rates": list(rates),
        "same_learning_rate_for_tt_and_dense": True,
        "local_steps_per_pair": local_steps,
        "evaluation_interval": interval,
        "max_gradient_evaluations": maximum,
        "min_delta": MIN_DELTA,
        "patience": PATIENCE,
        "selection_api": "update_validation_selection",
        "optimizer_global": "full_batch_simultaneous_gd",
        "optimizer_local": "joint_two_core_full_batch_gd_directional_svd",
        "gradient_batch_size": GRADIENT_BATCH_SIZE,
        "validation_batch_size": VALIDATION_BATCH_SIZE,
        "criterion": "mean_cross_entropy",
        "training_contract_version": TRAINING_CONTRACT_VERSION,
        "notebook_code_sha256": notebook_code_sha256(
            NOTEBOOK_PATH,
            excluded_assignments=("RUN_SMOKE", "RUN_MAIN_EXPERIMENT", "REGENERATE_DISPLAY_ARTIFACTS"),
        ),
        "implementation_sha256": implementation_sha256(),
    }


def training_settings(smoke):
    """学習 run ID の材料。Notebook 全コードの hash は含めない。"""
    settings = canonical_settings(smoke)
    settings.pop("notebook_code_sha256", None)
    return settings


def training_run_identifier(smoke):
    payload = json.dumps(training_settings(smoke), sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def run_identifier(smoke):
    return training_run_identifier(smoke)


def condition_directory(smoke, method, learning_rate):
    models_dir, _results_dir = experiment_dirs(smoke)
    rate = f"{learning_rate:.1f}".replace(".", "p")
    return models_dir / method / f"lr{rate}"


def publish_condition(directory, config, environment, run_state, history, updates, payloads):
    assert_writable(directory)

    def prepare(root):
        initial = root / "initial_model.pt"
        if not initial.is_file():
            shutil.copyfile(INITIAL_CHECKPOINT, initial)
        digest = file_sha256(initial)
        if digest != config["initial_model_sha256"] or digest != STAGE_C_SHA256:
            raise RuntimeError("条件の initial_model.pt が Stage C の checksum と一致しません。")

    def write_files(partial):
        atomic_torch_save(partial / "best_model.pt", payloads["best"])
        atomic_torch_save(partial / "final_model.pt", payloads["final"])
        atomic_write_csv(partial / "history.csv", history, HISTORY_FIELDS)
        atomic_write_csv(partial / "update_log.csv", updates, UPDATE_FIELDS)
        atomic_write_json(partial / "config.json", config)
        atomic_write_json(partial / "environment.json", environment)
        atomic_write_json(partial / "run_state.json", run_state)
        return [
            "best_model.pt", "final_model.pt", "history.csv", "update_log.csv",
            "config.json", "environment.json", "run_state.json",
        ]

    return publish_generation(
        directory,
        write_files,
        manifest_fields={
            "stop_reason": run_state["stop_reason"],
            "method": config["method"],
            "learning_rate": config["learning_rate"],
            "full_dataset_gradient_evaluations": run_state["full_dataset_gradient_evaluations"],
            "strict_best_loss": run_state["strict_best_loss"],
            "initial_model_sha256": config["initial_model_sha256"],
            "run_id": config["run_id"],
        },
        pointer_fields=(
            "stop_reason", "method", "learning_rate", "full_dataset_gradient_evaluations",
            "strict_best_loss", "run_id",
        ),
        prepare=prepare,
    )


def active_condition(directory, run_id):
    if not directory.exists():
        return None
    active = load_active_generation(
        directory,
        pinned_files={"initial_model.pt": "initial_model_sha256"},
        required_manifest_keys=("stop_reason", "run_id", "method", "learning_rate"),
    )
    if active is None:
        return None
    if active.manifest.get("run_id") != run_id:
        raise RuntimeError(
            "保存済み世代の run_id が現在の設定と違います。上書きせず停止します: "
            f"{directory} saved={active.manifest.get('run_id')} current={run_id}。"
            "実装や学習設定が変わった場合は、その条件ディレクトリを削除してから再実行してください。"
        )
    return active


## データと Stage C

学習データを読む処理は、実行スイッチが真のときだけ呼ぶ。Stage C は `initial_stage_c.pt` の bond 次元、mode、seed、site 0 の正準性を確認してから両方式の独立なコピーへ渡す。画像とラベルの checksum は、Notebook 06 が保存した値、または dtype と shape を含まない値の SHA-256 のどちらかが一致すれば採用し、どちらとも違う場合は停止する。


In [19]:
def value_sha256(tensor):
    values = tensor.detach().to("cpu").contiguous()
    array = values.numpy() if values.dtype != torch.uint8 else values.view(torch.uint8).numpy()
    return hashlib.sha256(array.tobytes()).hexdigest()


def require_cuda_device():
    if not torch.cuda.is_available():
        raise RuntimeError("Notebook 06 と同じ CUDA 実行が必要です。")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    return torch.device("cuda", torch.cuda.current_device())


def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def peak_memory(device):
    """条件開始後の max_memory_allocated。CUDA 以外は not_applicable、取得失敗は unverified。"""
    if device.type != "cuda":
        return None, "not_applicable"
    try:
        return int(torch.cuda.max_memory_allocated(device)), "recorded"
    except Exception:
        return None, "unverified"


def make_loader(dataset, batch_size):
    return DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False, num_workers=0)


def load_split():
    train_full, test_dataset = get_fashion_mnist_datasets(PROJECT_ROOT / "data", download=False)
    del test_dataset
    train_dataset, early_dataset, rank_dataset = split_fashion_mnist_dataset(
        train_full, SPLIT_LENGTHS, seed=SEED
    )
    if (len(train_dataset), len(early_dataset), len(rank_dataset)) != SPLIT_LENGTHS:
        raise RuntimeError("分割件数が 50,000 / 5,000 / 5,000 ではありません。")
    split_hash = "|".join(index_sha256(subset.indices) for subset in (train_dataset, early_dataset, rank_dataset))
    if split_hash != SPLIT_INDEX_SHA256:
        raise RuntimeError("split index の SHA-256 が Notebook 06 の記録と一致しません。")
    images = torch.stack([train_dataset[index][0] for index in range(len(train_dataset))]).contiguous()
    labels = torch.tensor([int(train_dataset[index][1]) for index in range(len(train_dataset))], dtype=torch.long)
    image_value = value_sha256(images)
    label_value = value_sha256(labels)
    image_full = tensor_sha256(images)
    label_full = tensor_sha256(labels)
    if image_value != IMAGE_SHA256 and image_full != IMAGE_SHA256:
        raise RuntimeError("学習画像の SHA-256 が Notebook 06 の記録と一致しません。")
    if label_value != LABEL_SHA256 and label_full != LABEL_SHA256:
        raise RuntimeError("学習ラベルの SHA-256 が Notebook 06 の記録と一致しません。")
    return {
        "train_dataset": train_dataset,
        "early_dataset": early_dataset,
        "rank_dataset": rank_dataset,
        "images": images,
        "labels": labels,
        "image_sha256_matched": image_value if image_value == IMAGE_SHA256 else image_full,
        "label_sha256_matched": label_value if label_value == LABEL_SHA256 else label_full,
    }


def payload_from_model(model, *, stage):
    return {
        "layer_name": TARGET_LAYER,
        "state_dict": {name: value.detach().cpu() for name, value in model.state_dict().items()},
        "out_modes": list(OUT_MODES),
        "in_modes": list(IN_MODES),
        "rank_cap": RANK_CAP,
        "stage": stage,
        "seed": SEED,
    }


def build_stage_c_model():
    if file_sha256(INITIAL_CHECKPOINT) != STAGE_C_SHA256:
        raise RuntimeError("initial_stage_c.pt の SHA-256 が Notebook 06 の記録と一致しません。")
    payload = torch.load(INITIAL_CHECKPOINT, map_location="cpu", weights_only=False)
    if payload.get("layer_name") != TARGET_LAYER or payload.get("stage") != "C":
        raise ValueError("Stage C の layer または stage が一致しません。")
    if tuple(payload.get("out_modes", ())) != OUT_MODES or tuple(payload.get("in_modes", ())) != IN_MODES:
        raise ValueError("Stage C の tensorization が一致しません。")
    if int(payload.get("rank_cap", -1)) != RANK_CAP or int(payload.get("seed", -1)) != SEED:
        raise ValueError("Stage C の rank 上限または seed が一致しません。")
    if int(payload.get("canonical_center", -1)) != 0:
        raise ValueError("Stage C の正準中心が site 0 ではありません。")
    state = payload["state_dict"]
    prefix = f"{TARGET_LAYER}.cores."
    keys = sorted((key for key in state if key.startswith(prefix)), key=lambda key: int(key.rsplit(".", 1)[-1]))
    cores = [state[key].detach().to(dtype=torch.float32, device="cpu") for key in keys]
    ranks = tuple(int(rank) for rank in tt_matrix_ranks(cores))
    if ranks != EXPECTED_ACTUAL_RANKS:
        raise ValueError(f"Stage C の bond 次元が一致しません: {ranks}")
    errors = tt_matrix_canonicality_errors(cores, OUT_MODES, IN_MODES, 0)
    if errors["left"] or max(float(item) for item in errors["right"]) > 1e-4:
        raise ValueError("Stage C が site 0 中心の正準形ではありません。")
    model = MNISTMLP()
    model.fc1 = TTLinear(cores, OUT_MODES, IN_MODES, bias=state[f"{TARGET_LAYER}.bias"].detach())
    own = dict(model.named_parameters())
    dense_names = tuple(name for name in own if not name.startswith(prefix))
    if dense_names != EXPECTED_DENSE_PARAMETERS:
        raise ValueError(f"通常 parameter が想定と違います: {dense_names}")
    with torch.no_grad():
        for key, value in state.items():
            if key.startswith(prefix) or key == f"{TARGET_LAYER}.bias":
                continue
            own[key].copy_(value)
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    return model


def bond_ranks(model):
    return tuple(int(rank) for rank in tt_matrix_ranks([core.detach().cpu() for core in model.fc1.cores]))


def core_shapes(model):
    return tuple(tuple(int(size) for size in core.shape) for core in model.fc1.cores)


## 1 ブロックの学習

全コア同時GDは `update_tt_linear_global_full_batch` を評価間隔の回数だけ繰り返す。DMRG-like（2コア局所更新）は同じ回数を 1 往復として `train_scoped_full_batch_budget` に渡す。どちらもブロックのあとで early-stopping validation を評価し、`update_validation_selection` を進める。方式に無い更新回数は 0 にせず `not_applicable` にする。

学習時間は parameter 更新に使った秒だけを積む。validation 評価の秒は別列である。`train_scoped_full_batch_budget` は各往復の開始時と終了時を評価するため、DMRG-like（2コア局所更新）ではブロック境界の評価が重複する。性能対学習時間の図は `cumulative_train_seconds` を使う。wall-clock time、total evaluation time、end-to-end の総費用には、この重複した評価秒を含む。CUDA ではブロックの前後で synchronize し、条件の開始時に peak memory をリセットする。


In [20]:
def environment_record(device):
    return {
        "dtype": "float32",
        "device": str(device),
        "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else "cpu",
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "allow_tf32_matmul": bool(torch.backends.cuda.matmul.allow_tf32),
        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
        "execution": "eager",
    }


def evaluate_pair(model, train_loader, validation_loader, device):
    criterion = nn.CrossEntropyLoss(reduction="mean")
    synchronize(device)
    started = time.perf_counter()
    train_loss, train_accuracy = evaluate(model, train_loader, criterion, device)
    validation_loss, validation_accuracy = evaluate(model, validation_loader, criterion, device)
    synchronize(device)
    elapsed = time.perf_counter() - started
    values = (train_loss, train_accuracy, validation_loss, validation_accuracy)
    if not all(math.isfinite(value) for value in values):
        raise NonFiniteLocalUpdate("評価指標が有限ではありません。")
    return {
        "train_loss": float(train_loss),
        "train_accuracy": float(train_accuracy),
        "validation_loss": float(validation_loss),
        "validation_accuracy": float(validation_accuracy),
        "seconds": elapsed,
    }


def counts_for(method, evaluations):
    if method == GLOBAL_METHOD:
        return {"global": (evaluations, "recorded"), "local": (None, "not_applicable")}
    if method == LOCAL_METHOD:
        return {"local": (evaluations, "recorded"), "global": (None, "not_applicable")}
    raise ValueError(f"未知の方式です: {method}")



class _BlockFailure(Exception):
    def __init__(self, reason, detail, rows, train_seconds, eval_seconds):
        super().__init__(detail)
        self.reason = reason
        self.detail = detail
        self.rows = rows
        self.train_seconds = train_seconds
        self.eval_seconds = eval_seconds


def snapshot_parameters(model):
    return {name: parameter.detach().cpu().clone() for name, parameter in model.named_parameters()}


def restore_parameters(model, snapshot):
    """CPU 上の整合 snapshot へ戻す。失敗理由を返し、新たな例外では止めない。"""
    try:
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
        model.to("cpu")
        with torch.no_grad():
            named = dict(model.named_parameters())
            for name, value in snapshot.items():
                named[name].copy_(value)
        return None
    except Exception as exc:
        return f"{type(exc).__name__}: {exc}"


def model_from_snapshot(snapshot):
    restored = build_stage_c_model()
    with torch.no_grad():
        named = dict(restored.named_parameters())
        for name, value in snapshot.items():
            named[name].copy_(value)
    return restored


def train_condition(method, learning_rate, bundle, device, smoke):
    interval = SMOKE_EVALUATION_INTERVAL if smoke else EVALUATION_INTERVAL
    maximum = SMOKE_MAX_GRADIENT_EVALUATIONS if smoke else MAX_GRADIENT_EVALUATIONS
    local_steps = SMOKE_LOCAL_STEPS_PER_PAIR if smoke else LOCAL_STEPS_PER_PAIR
    if interval != 2 * (N_CORES - 1) * local_steps:
        raise RuntimeError("この実行の評価間隔が設定式と一致しません。")
    run_id = run_identifier(smoke)
    directory = condition_directory(smoke, method, learning_rate)
    directory.mkdir(parents=True, exist_ok=True)
    active = active_condition(directory, run_id)
    if active is not None and active.manifest["stop_reason"] in FINISHED_STOPS:
        return {"skipped": True, "directory": directory, "stop_reason": active.manifest["stop_reason"]}

    model = build_stage_c_model()
    if active is not None:
        load_tt_linear_sweep_checkpoint(active.path / "final_model.pt", model)
        history = [coerce_row(row) for row in read_table(active.path / "history.csv")]
        updates = [coerce_row(row) for row in read_table(active.path / "update_log.csv")]
        selection = selection_from_history(history)
        if selection.early_stop_triggered:
            raise RuntimeError("early stopping 済みの履歴が running として残っています。")
        train_seconds = float(history[-1]["cumulative_train_seconds"])
        eval_seconds = float(history[-1]["cumulative_eval_seconds"])
        best_model = build_stage_c_model()
        load_tt_linear_sweep_checkpoint(active.path / "best_model.pt", best_model)
    else:
        history = []
        updates = []
        selection = None
        train_seconds = 0.0
        eval_seconds = 0.0
        best_model = None
    model.to(device)
    if best_model is not None:
        best_model.to(device)
    images = bundle["images"].to(device)
    labels = bundle["labels"].to(device)
    source = GpuResidentTensorBatches(images, labels)
    train_loader = make_loader(bundle["train_dataset"], VALIDATION_BATCH_SIZE)
    validation_loader = make_loader(bundle["early_dataset"], VALIDATION_BATCH_SIZE)
    diagnostic_images = bundle["images"][:DIAGNOSTIC_BATCH_SIZE].contiguous()
    diagnostic_labels = bundle["labels"][:DIAGNOSTIC_BATCH_SIZE].contiguous()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    initial_payload = payload_from_model(build_stage_c_model(), stage="C")
    config = canonical_settings(smoke)
    config.update({
        "run_id": run_id,
        "method": method,
        "learning_rate": learning_rate,
        "initial_model_sha256": STAGE_C_SHA256,
        "reuse_decision": "new_run",
        "reaggregated": False,
        "source_notebook": None,
        "source_run_id": None,
        "source_checkpoint": None,
    })
    environment = environment_record(device)
    shapes = core_shapes(model)
    wall_offset = wall_clock_offset(history)
    wall_started = time.perf_counter()


    failed_attempt = None
    stay_on_cpu = False
    last_consistent = None

    def publish(status, detail):
        evaluations = 0 if not history else int(history[-1]["full_dataset_gradient_evaluations"])
        memory, memory_status = peak_memory(device)
        run_state = {
            "stop_reason": status,
            "stop_detail": detail,
            "full_dataset_gradient_evaluations": evaluations,
            "strict_best_loss": None if selection is None else selection.strict_best_loss,
            "strict_best_step": None if selection is None else selection.strict_best_step,
            "seconds_train": train_seconds,
            "seconds_eval": eval_seconds,
            "peak_gpu_memory_bytes": memory,
            "peak_gpu_memory_status": memory_status,
            "peak_memory_scope": PEAK_MEMORY_SCOPE,
            "timing_scope": TIMING_SCOPE,
            "history_rows": len(history),
            "failed_attempt": failed_attempt,
        }
        if failed_attempt is not None:
            if last_consistent is None:
                raise RuntimeError("戻す評価ブロック境界がありません。")
            current = payload_from_model(model_from_snapshot(last_consistent["parameters"]), stage="trained")
        else:
            current = payload_from_model(model.cpu(), stage="trained")
            if not stay_on_cpu:
                model.to(device)
        if best_model is None:
            best_payload = current
        else:
            best_payload = payload_from_model(best_model.cpu(), stage="trained")
            if failed_attempt is None and not stay_on_cpu:
                best_model.to(device)
        publish_condition(
            directory, config, environment, run_state, history, updates,
            {"initial": initial_payload, "best": best_payload, "final": current},
        )

    if not history:
        metrics = evaluate_pair(model, train_loader, validation_loader, device)
        eval_seconds += metrics["seconds"]
        selection = initial_validation_selection(metrics["validation_loss"], 0)
        best_model = build_stage_c_model().to(device)
        metric_values = missing_metrics()
        metric_values.update(metrics)
        history.append(history_row(
            method, learning_rate, 0, 0, selection, metric_values,
            {"train": train_seconds, "eval": eval_seconds, "wall": resumed_wall_clock_seconds(wall_offset, wall_started, time.perf_counter())},
            bond_ranks(model), counts_for(method, 0),
        ))
        require_monotonic_wall_clock(history)
        publish("running", "初期状態を評価した。")


    def fail_block(reason, detail, rows, attempt_train, attempt_eval, boundary, oom):
        nonlocal failed_attempt, train_seconds, eval_seconds, stop_reason, stop_detail, stay_on_cpu
        if oom:
            stay_on_cpu = True
        restore_error = restore_parameters(model, boundary["parameters"])
        del updates[boundary["update_count"]:]
        train_seconds = boundary["train_seconds"]
        eval_seconds = boundary["eval_seconds"]
        failed_attempt = {
            "stop_reason": reason,
            "stop_detail": detail,
            "rolled_back_to_gradient_evaluations": int(history[-1]["full_dataset_gradient_evaluations"]),
            "attempted_updates": len(rows),
            "train_seconds": attempt_train,
            "eval_seconds": attempt_eval,
            "restore_error": restore_error,
            "updates": rows,
        }
        stop_reason = reason
        stop_detail = detail

    stop_reason = None
    stop_detail = ""
    while int(history[-1]["full_dataset_gradient_evaluations"]) < maximum and not selection.early_stop_triggered:
        boundary = {
            "parameters": snapshot_parameters(model),
            "update_count": len(updates),
            "train_seconds": train_seconds,
            "eval_seconds": eval_seconds,
        }
        last_consistent = boundary
        try:
            synchronize(device)
            started = time.perf_counter()
            block_values = {name: [] for name in (
                "core_gradient_norm", "dense_gradient_norm", "joint_gradient_norm",
                "core_relative_update", "dense_relative_update", "joint_relative_update",
            )}
            if method == GLOBAL_METHOD:
                previous = int(history[-1]["full_dataset_gradient_evaluations"])
                for step_index in range(1, interval + 1):
                    step_started = time.perf_counter()
                    result = update_tt_linear_global_full_batch(
                        model, source, layer_name=TARGET_LAYER, learning_rate=learning_rate,
                        batch_size=GRADIENT_BATCH_SIZE, device=device,
                    )
                    if result.core_shapes != shapes:
                        raise RuntimeError("Global 更新で core shape が変わりました。")
                    done = previous + step_index
                    updates.append({
                        "method": method, "learning_rate": learning_rate,
                        "full_dataset_gradient_evaluations": done,
                        "pair_index": None, "direction": None, "local_step": None,
                        "center_before": None, "center_after": None, "retained_rank": None,
                        "discarded_squared_norm": None,
                        "joint_gradient_norm": result.gradient_norm,
                        "core_gradient_norm": result.core_gradient_norm,
                        "dense_gradient_norm": result.dense_gradient_norm,
                        "joint_relative_update": result.relative_update,
                        "core_relative_update": result.core_relative_update,
                        "dense_relative_update": result.dense_relative_update,
                        "full_batch_loss": result.full_batch_loss,
                        "update_seconds": time.perf_counter() - step_started,
                        "finite": True,
                    })
                    for name, value in (
                        ("joint_gradient_norm", result.gradient_norm),
                        ("core_gradient_norm", result.core_gradient_norm),
                        ("dense_gradient_norm", result.dense_gradient_norm),
                        ("joint_relative_update", result.relative_update),
                        ("core_relative_update", result.core_relative_update),
                        ("dense_relative_update", result.dense_relative_update),
                    ):
                        block_values[name].append(value)
            else:
                # この API は終了時に best へ戻さない。次の往復へ渡る model は final。
                result = train_scoped_full_batch_budget(
                    model, source, train_loader, validation_loader,
                    scope=TTUpdateScope("joint"), layer_name=TARGET_LAYER,
                    batch_size=GRADIENT_BATCH_SIZE, total_updates=interval,
                    evaluation_interval=interval, rank_cap=RANK_CAP,
                    tt_learning_rate=learning_rate, dense_learning_rate=learning_rate,
                    diagnostic_images=diagnostic_images, diagnostic_labels=diagnostic_labels,
                    device=device, local_steps_per_pair=local_steps, max_sweeps=1, center=0,
                )
                if result.stop_reason != "max_updates":
                    raise _BlockFailure(
                        result.stop_reason, result.stop_detail, list(result.local_updates),
                        float(result.seconds_train), float(result.seconds_eval),
                    )
                if result.center != 0:
                    raise RuntimeError("1 往復後の正準中心が site 0 に戻っていません。")
                if len(result.history) != 2 or int(result.history[1]["cumulative_updates"]) != interval:
                    raise RuntimeError("1 往復の終了評価が、引き継ぐ final の更新数と一致しません。")
                previous = int(history[-1]["full_dataset_gradient_evaluations"])
                for entry in result.local_updates:
                    done = previous + int(entry["cumulative_updates"])
                    updates.append({
                        "method": method, "learning_rate": learning_rate,
                        "full_dataset_gradient_evaluations": done,
                        "pair_index": entry["pair_index"], "direction": entry["direction"],
                        "local_step": entry["local_step"], "center_before": entry["center_before"],
                        "center_after": entry["center_after"], "retained_rank": entry["retained_rank"],
                        "discarded_squared_norm": entry["discarded_squared_norm"],
                        "joint_gradient_norm": None,
                        "core_gradient_norm": entry["tt_gradient_norm"],
                        "dense_gradient_norm": entry["dense_gradient_norm"],
                        "joint_relative_update": None,
                        "core_relative_update": entry["tt_relative_update"],
                        "dense_relative_update": entry["dense_relative_update"],
                        "full_batch_loss": entry["full_batch_loss"],
                        "update_seconds": entry["update_seconds"],
                        "finite": entry["finite"],
                    })
                    block_values["core_gradient_norm"].append(entry["tt_gradient_norm"])
                    block_values["dense_gradient_norm"].append(entry["dense_gradient_norm"])
                    block_values["core_relative_update"].append(entry["tt_relative_update"])
                    block_values["dense_relative_update"].append(entry["dense_relative_update"])
                train_seconds += float(result.seconds_train)
                point = result.history[1]
                metrics = {
                    "train_loss": float(point["train_loss"]),
                    "train_accuracy": float(point["train_accuracy"]),
                    "validation_loss": float(point["early_stopping_validation_loss"]),
                    "validation_accuracy": float(point["early_stopping_validation_accuracy"]),
                    "seconds": float(result.seconds_eval),
                }
            synchronize(device)
            if method == GLOBAL_METHOD:
                train_seconds += time.perf_counter() - started
                metrics = evaluate_pair(model, train_loader, validation_loader, device)
            eval_seconds += metrics["seconds"]
            evaluations = int(history[-1]["full_dataset_gradient_evaluations"]) + interval
            selection = update_validation_selection(
                selection, metrics["validation_loss"], evaluations, min_delta=MIN_DELTA, patience=PATIENCE,
            )
            if selection.strict_best_step == evaluations:
                best_model = copy.deepcopy(model)
            metric_values = {}
            for name, values in block_values.items():
                if method == LOCAL_METHOD and name.startswith("joint"):
                    metric_values[name] = (None, "not_applicable")
                elif method == GLOBAL_METHOD or values:
                    metric_values[name] = mean_or_missing(values)
                else:
                    metric_values[name] = (None, "not_recorded")
            metric_values.update(metrics)
            history.append(history_row(
                method, learning_rate, len(history), evaluations, selection, metric_values,
                {"train": train_seconds, "eval": eval_seconds, "wall": resumed_wall_clock_seconds(wall_offset, wall_started, time.perf_counter())},
                bond_ranks(model), counts_for(method, evaluations),
            ))
            require_monotonic_wall_clock(history)
            if selection.early_stop_triggered:
                stop_reason = "early_stopping"
                stop_detail = (
                    f"validation loss が min_delta={MIN_DELTA} 以上改善しない観測が {PATIENCE} 回続いた。"
                    "収束を意味しない。"
                )
                break
            publish("running", "評価間隔の終了時点を保存した。")
        except _BlockFailure as exc:
            fail_block(exc.reason, exc.detail, exc.rows, exc.train_seconds, exc.eval_seconds, boundary, exc.reason == "out_of_memory")
            break
        except NonFiniteLocalUpdate as exc:
            attempted = [dict(row) for row in updates[boundary["update_count"]:]]
            attempt_train = sum(float(row["update_seconds"]) for row in attempted)
            fail_block("non_finite", str(exc), attempted, attempt_train, 0.0, boundary, False)
            break
        except torch.cuda.OutOfMemoryError as exc:
            attempted = [dict(row) for row in updates[boundary["update_count"]:]]
            attempt_train = sum(float(row["update_seconds"]) for row in attempted)
            fail_block("out_of_memory", str(exc), attempted, attempt_train, 0.0, boundary, True)
            break
    if stop_reason is None:
        stop_reason = "max_gradient_evaluations"
        stop_detail = (
            f"最大 full-dataset gradient evaluation 数 {maximum} に達した。収束を意味しない。"
        )
    publish(stop_reason, stop_detail)

    return {"skipped": False, "directory": directory, "stop_reason": stop_reason}


## 既存結果の監査

判定は実行時に保存ファイルを読んで行う。Notebook 02 は early stopping が無効で、patience 1、`min_delta` 0、local step 1 または 4、最大往復 5 または 20 なので参考値に分ける。Notebook 03 は学習率 0.2 と 0.3、local step 8 だが、patience 1、`min_delta` 0、最大 40 往復なので、今回の停止規則の履歴としては足りない。Notebook 05 の joint は学習率 0.3 の 40 往復で early stopping が無効であり、Notebook 06 がその続きを持っている。Notebook 06 の通常 TT は Adam、batch 64、学習率 0.001 なので全コア同時GDではない。

Notebook 06 の DMRG-like は読み取り専用で、上書きも削除もしない。履歴構造と停止規則は監査できるが、保存時の学習実装と現行実装の同一性は確認できない。そのため Notebook 06 の結果は参考値として `reuse_audit.csv` に残し、主比較の表へは混ぜない。Notebook 11 の主比較用 DMRG-like（2コア局所更新）、学習率 0.3 は新規実行する。


In [21]:
def reference_row(notebook, decision, reason, reaggregated=False):
    return {
        "source_notebook": notebook,
        "source_run_id": None,
        "source_checkpoint": None,
        "reuse_decision": decision,
        "reaggregated": reaggregated,
        "reason": reason,
    }



def notebook06_history_issues(history, run_state, log_rows):
    """保存物から確認できる履歴の欠落だけを返す。実装 hash や測定範囲はここでは判定しない。"""
    issues = []
    if not history:
        return ["history_missing"]
    updates = [int(float(row["cumulative_local_updates"])) for row in history]
    if updates[0] != 0:
        issues.append("history_start")
    if updates != sorted(updates) or len(updates) != len(set(updates)):
        issues.append("history_order")
    if any(right - left != EVALUATION_INTERVAL for left, right in zip(updates, updates[1:])):
        issues.append("history_interval")
    if log_rows is None:
        issues.append("update_log_missing")
    else:
        counts = [int(float(row["cumulative_updates"])) for row in log_rows]
        if counts != list(range(1, len(counts) + 1)):
            issues.append("update_log_sequence")
        elif not counts or counts[-1] != updates[-1]:
            issues.append("update_log_matches_history")
        if any(row.get("scope") != "joint" for row in log_rows):
            issues.append("update_scope")
        n_cores = len(OUT_MODES)
        visits = [(index, "right") for index in range(n_cores - 1)]
        visits += [(index, "left") for index in range(n_cores - 2, -1, -1)]
        cycle = [(pair, direction, step) for pair, direction in visits for step in range(LOCAL_STEPS_PER_PAIR)]
        for index, row in enumerate(log_rows):
            expected = cycle[index % len(cycle)]
            actual = (int(float(row["pair_index"])), row["direction"], int(float(row["local_step"])))
            if actual != expected:
                issues.append("sweep_order")
                break
    if not (NOTEBOOK06_DMRG / "best_model.pt").is_file() or not (NOTEBOOK06_DMRG / "final_model.pt").is_file():
        issues.append("checkpoint_missing")
    best_step = int(float(history[-1]["strict_best_step"]))
    if best_step not in updates:
        issues.append("best_step_not_in_history")
    recorded = run_state.get("total_local_updates", run_state.get("optimizer_steps"))
    if recorded is None or int(recorded) != updates[-1]:
        issues.append("run_state_updates")
    return issues


def classify_notebook06_local():
    config_path = NOTEBOOK06_DMRG / "config.json"
    history_path = NOTEBOOK06_DMRG / "history.csv"
    state_path = NOTEBOOK06_DMRG / "run_state.json"
    if not config_path.is_file() or not history_path.is_file() or not state_path.is_file():
        return "new_run", "reference_only", "Notebook 06 の DMRG-like 成果物がありません。", None
    config = json.loads(config_path.read_text(encoding="utf-8"))
    history = read_table(history_path)
    run_state = json.loads(state_path.read_text(encoding="utf-8"))
    checks = {
        "learning_rate": float(config.get("tt_learning_rate", -1)) == 0.3 and float(config.get("dense_parameter_learning_rate", -1)) == 0.3,
        "local_steps": int(config.get("local_steps_per_pair", -1)) == LOCAL_STEPS_PER_PAIR,
        "interval": int(config.get("evaluation_interval", -1)) == EVALUATION_INTERVAL,
        "budget": int(config.get("max_total_local_updates", -1)) == MAX_GRADIENT_EVALUATIONS,
        "min_delta": float(config.get("min_delta", -1)) == MIN_DELTA,
        "patience": int(config.get("patience", -1)) == PATIENCE,
        "checkpoint": config.get("initial_checkpoint_sha256") == STAGE_C_SHA256,
        "split": config.get("split_index_sha256") == SPLIT_INDEX_SHA256,
        "image": config.get("image_sha256") == IMAGE_SHA256,
        "label": config.get("label_sha256") == LABEL_SHA256,
        "modes": tuple(config.get("out_modes", ())) == OUT_MODES and tuple(config.get("in_modes", ())) == IN_MODES,
        "ranks": tuple(config.get("expected_actual_ranks", ())) == EXPECTED_ACTUAL_RANKS,
        "criterion": config.get("criterion") == "mean_cross_entropy",
        "selection": config.get("selection_rule") == "strict_best_and_stopping_reference",
        "batch": int(config.get("batch_size", -1)) == GRADIENT_BATCH_SIZE,
        "rank_cap": int(config.get("rank_cap", -1)) == RANK_CAP,
    }
    state = initial_validation_selection(
        float(history[0]["early_stopping_validation_loss"]), int(float(history[0]["cumulative_local_updates"]))
    )
    triggered = None
    for index, row in enumerate(history[1:], start=1):
        state = update_validation_selection(
            state, float(row["early_stopping_validation_loss"]), int(float(row["cumulative_local_updates"])),
            min_delta=MIN_DELTA, patience=PATIENCE,
        )
        if state.early_stop_triggered:
            triggered = index
            break
    checks["replay_end"] = triggered == len(history) - 1
    checks["best_step"] = state.strict_best_step == int(float(history[-1]["strict_best_step"]))
    checks["best_loss"] = math.isclose(state.strict_best_loss, float(history[-1]["strict_best_loss"]), rel_tol=0.0, abs_tol=1e-12)
    checks["stop_reason"] = run_state.get("stop_reason") == "early_stopping"
    checks["gradient_passes"] = int(float(history[-1]["train_full_dataset_gradient_passes"])) == int(float(history[-1]["cumulative_local_updates"]))
    log_rows = read_table(NOTEBOOK06_DMRG / "local_updates.csv") if (NOTEBOOK06_DMRG / "local_updates.csv").is_file() else None
    for name in notebook06_history_issues(history, run_state, log_rows):
        checks[name] = False
    failed = [name for name, passed in checks.items() if not passed]
    source_run = f"{config.get('notebook_code_sha256')}:{config.get('implementation_sha256')}"
    if failed:
        return "new_run", "reference_only", "再利用条件を満たさない項目: " + ", ".join(failed), source_run
    saved_initial = torch.load(NOTEBOOK06_DMRG / "initial_model.pt", map_location="cpu", weights_only=False)
    stage_initial = torch.load(INITIAL_CHECKPOINT, map_location="cpu", weights_only=False)
    saved_state = saved_initial.get("state_dict", {})
    stage_state = stage_initial.get("state_dict", {})
    same_initial = set(saved_state) == set(stage_state) and all(torch.equal(stage_state[key], saved_state[key]) for key in stage_state)
    if not same_initial:
        return "new_run", "reference_only", "Notebook 06 の初期 state が Stage C と一致しません。", source_run
    recorded_hash = config.get("implementation_sha256")
    current_hash = notebook06_implementation_sha256()
    if recorded_hash != current_hash:
        return (
            "new_run",
            "reference_only",
            "履歴構造と停止規則は監査できたが、保存時の学習実装と現行実装の同一性を確認できないため、"
            "Notebook 06の結果は参考値に限定し、Notebook 11の主比較用Local lr=0.3は新規実行する。",
            source_run,
        )
    return (
        "reuse_reaggregate",
        "reuse_reaggregate",
        "Notebook 06 の local sweep、学習率 0.3 を再集計する。"
        "peak memory の測定範囲は Notebook 11 と同一とは確認していない。best checkpoint の重みは再評価していないので unverified。",
        source_run,
    )


def audit_rows():
    rows = [
        reference_row(
            "02_full_batch_sweep_update_budget_grid", "reference_only",
            "early stopping は無効。patience 1、min_delta 0、local step は 1 または 4、最大往復は 5 または 20。学習率 0.1 という値だけを今回の候補に使う。",
        ),
        reference_row(
            "03_joint_learning_rate_02_03_s40_l8", "reference_only",
            "学習率 0.2 と 0.3、local step 8 だが、patience 1、min_delta 0、最大 40 往復であり、今回の停止規則より前で履歴が終わる。",
        ),
        reference_row(
            "05_update_scope_ablation_gpu_resident", "reference_only",
            "joint、学習率 0.3、local step 8 の 40 往復は Notebook 06 の先頭に含まれる。early stopping は無効なので、単独の主比較には使わない。",
        ),
        reference_row(
            "06_dense_tt_dmrg_like_convergence_comparison/standard_tt", "reference_only",
            "通常 TT は Adam、batch 64、学習率 0.001 であり、全データの単純 GD ではない。",
        ),
    ]
    decision, source_decision, reason, source_run = classify_notebook06_local()
    row = reference_row(NOTEBOOK06_NAME + "/dmrg_like", source_decision, reason, reaggregated=decision == "reuse_reaggregate")
    row["source_run_id"] = source_run
    row["source_checkpoint"] = str((NOTEBOOK06_DMRG / "best_model.pt").relative_to(PROJECT_ROOT))
    rows.append(row)
    for method in METHODS:
        for rate in LEARNING_RATES:
            if method == LOCAL_METHOD and rate == 0.3 and decision == "reuse_reaggregate":
                continue
            rows.append({
                "source_notebook": None,
                "source_run_id": None,
                "source_checkpoint": None,
                "reuse_decision": "new_run",
                "reaggregated": False,
                "reason": f"{method} 学習率 {rate} は一致する保存履歴がないため新規実行する。",
                "method": method,
                "learning_rate": rate,
            })
    return rows, decision


def reaggregate_notebook06(run_id, source_run):
    directory = condition_directory(False, LOCAL_METHOD, 0.3)
    directory.mkdir(parents=True, exist_ok=True)
    active = active_condition(directory, run_id)
    if active is not None and active.manifest["stop_reason"] in FINISHED_STOPS:
        return directory
    history06 = read_table(NOTEBOOK06_DMRG / "history.csv")
    updates06 = read_table(NOTEBOOK06_DMRG / "local_updates.csv")
    grouped = {}
    for row in updates06:
        evaluation = int(float(row["cumulative_updates"]))
        block = (evaluation + EVALUATION_INTERVAL - 1) // EVALUATION_INTERVAL
        grouped.setdefault(block, []).append(row)
    history = []
    updates = []
    selection = None
    for row in history06:
        evaluations = int(float(row["cumulative_local_updates"]))
        if selection is None:
            selection = initial_validation_selection(float(row["early_stopping_validation_loss"]), evaluations)
            metrics = missing_metrics()
        else:
            selection = update_validation_selection(
                selection, float(row["early_stopping_validation_loss"]), evaluations,
                min_delta=MIN_DELTA, patience=PATIENCE,
            )
            block_rows = grouped.get(evaluations // EVALUATION_INTERVAL, [])
            metrics = {}
            for source_name, target_name in (
                ("tt_gradient_norm", "core_gradient_norm"),
                ("dense_gradient_norm", "dense_gradient_norm"),
                ("tt_relative_update", "core_relative_update"),
                ("dense_relative_update", "dense_relative_update"),
            ):
                metrics[target_name] = mean_or_missing([item[source_name] for item in block_rows])
            metrics["joint_gradient_norm"] = (None, "not_applicable")
            metrics["joint_relative_update"] = (None, "not_applicable")
        metrics.update({
            "train_loss": float(row["train_loss"]),
            "train_accuracy": float(row["train_accuracy"]),
            "validation_loss": float(row["early_stopping_validation_loss"]),
            "validation_accuracy": float(row["early_stopping_validation_accuracy"]),
        })
        history.append(history_row(
            LOCAL_METHOD, 0.3, int(float(row["sweep"])), evaluations, selection, metrics,
            {
                "train": float(row["cumulative_train_seconds"]),
                "eval": float(row["cumulative_eval_seconds"]),
                "wall": float(row["wall_clock_seconds"]),
            },
            tuple(int(rank) for rank in row["actual_ranks"].split("|")),
            counts_for(LOCAL_METHOD, evaluations),
        ))
    for row in updates06:
        updates.append({
            "method": LOCAL_METHOD, "learning_rate": 0.3,
            "full_dataset_gradient_evaluations": int(float(row["cumulative_updates"])),
            "pair_index": int(float(row["pair_index"])), "direction": row["direction"],
            "local_step": int(float(row["local_step"])),
            "center_before": int(float(row["center_before"])),
            "center_after": int(float(row["center_after"])),
            "retained_rank": int(float(row["retained_rank"])),
            "discarded_squared_norm": float(row["discarded_squared_norm"]),
            "joint_gradient_norm": None,
            "core_gradient_norm": float(row["tt_gradient_norm"]),
            "dense_gradient_norm": float(row["dense_gradient_norm"]),
            "joint_relative_update": None,
            "core_relative_update": float(row["tt_relative_update"]),
            "dense_relative_update": float(row["dense_relative_update"]),
            "full_batch_loss": float(row["full_batch_loss"]),
            "update_seconds": float(row["update_seconds"]),
            "finite": str(row["finite"]).lower() == "true",
        })
    config = canonical_settings(False)
    config.update({
        "run_id": run_id, "method": LOCAL_METHOD, "learning_rate": 0.3,
        "initial_model_sha256": STAGE_C_SHA256, "reuse_decision": "reuse_reaggregate",
        "reaggregated": True, "source_notebook": NOTEBOOK06_NAME + "/dmrg_like",
        "source_run_id": source_run, "source_checkpoint": str((NOTEBOOK06_DMRG / "best_model.pt").relative_to(PROJECT_ROOT)),
    })
    run_state06 = json.loads((NOTEBOOK06_DMRG / "run_state.json").read_text(encoding="utf-8"))
    run_state = {
        "stop_reason": "early_stopping",
        "stop_detail": "Notebook 06 の保存履歴を update_validation_selection で再生した。",
        "full_dataset_gradient_evaluations": int(history[-1]["full_dataset_gradient_evaluations"]),
        "strict_best_loss": selection.strict_best_loss,
        "strict_best_step": selection.strict_best_step,
        "seconds_train": float(history[-1]["cumulative_train_seconds"]),
        "seconds_eval": float(history[-1]["cumulative_eval_seconds"]),
        "peak_gpu_memory_bytes": run_state06.get("peak_training_allocated_bytes"),
        "peak_gpu_memory_status": "recorded_by_notebook06_scope_unverified",
        "timing_scope": "notebook06_recorded_not_remeasured",
        "peak_memory_scope": "notebook06_recorded_scope_unverified",
        "history_rows": len(history),
    }
    payloads = {
        "initial": torch.load(NOTEBOOK06_DMRG / "initial_model.pt", map_location="cpu", weights_only=False),
        "best": torch.load(NOTEBOOK06_DMRG / "best_model.pt", map_location="cpu", weights_only=False),
        "final": torch.load(NOTEBOOK06_DMRG / "final_model.pt", map_location="cpu", weights_only=False),
    }
    publish_condition(
        directory, config, {"source": "notebook06_reaggregate"}, run_state, history, updates, payloads,
    )
    return directory


## smoke

smoke は学習率 0.3 だけ、local step 1、したがって評価間隔 10 の 1 ブロックだけを両方式で行う。確認するのは、同じ初期 state、全コア同時GDでは全 parameter が更新され core shape が変わらないこと、DMRG-like（2コア局所更新）では対象 pair と通常 parameter が更新され対象外 core がその 1 步の前後で変わらないこと、retained rank が上限以下であること、非有限損失で parameter が書き変わらないこと、保存した世代を読み直して完了済みなら再学習しないことである。smoke のディレクトリは main と別であり、main の初期状態には使わない。


In [22]:
def assert_states_equal(left, right):
    left_state = left.state_dict()
    right_state = right.state_dict()
    if list(left_state) != list(right_state):
        raise RuntimeError("初期 state の parameter 名が一致しません。")
    for name in left_state:
        if not torch.equal(left_state[name], right_state[name]):
            raise RuntimeError(f"初期 state が一致しません: {name}")


def smoke_nonfinite():
    class NanLoss(nn.Module):
        def __init__(self):
            super().__init__()
            self.reduction = "mean"

        def forward(self, logits, labels):
            return torch.full((), float("nan"), dtype=logits.dtype)

    probe = build_stage_c_model()
    generator = torch.Generator().manual_seed(0)
    images = torch.rand(2, 1, 28, 28, generator=generator)
    labels = torch.zeros(2, dtype=torch.long)
    loader = DataLoader(torch.utils.data.TensorDataset(images, labels), batch_size=2, shuffle=False)
    before = {name: parameter.detach().clone() for name, parameter in probe.named_parameters()}
    try:
        update_tt_linear_global_full_batch(
            probe, loader, layer_name=TARGET_LAYER, learning_rate=0.3, batch_size=None,
            device=torch.device("cpu"), criterion=NanLoss(),
        )
    except NonFiniteLocalUpdate:
        detected = True
    else:
        detected = False
    if not detected:
        raise RuntimeError("非有限損失を検出できませんでした。")
    for name, parameter in probe.named_parameters():
        if not torch.equal(parameter, before[name]):
            raise RuntimeError("非有限値を検出したあとに parameter が変わりました。")


def smoke_one_step(bundle, device):
    global_model = build_stage_c_model().to(device)
    local_model = build_stage_c_model().to(device)
    assert_states_equal(global_model.cpu(), local_model.cpu())
    global_model.to(device)
    local_model.to(device)
    source = GpuResidentTensorBatches(bundle["images"].to(device), bundle["labels"].to(device))
    before = {name: parameter.detach().clone() for name, parameter in global_model.named_parameters()}
    shapes = core_shapes(global_model)
    result = update_tt_linear_global_full_batch(
        global_model, source, layer_name=TARGET_LAYER, learning_rate=0.3,
        batch_size=GRADIENT_BATCH_SIZE, device=device,
    )
    if result.core_shapes != shapes:
        raise RuntimeError("Global の 1 步で core shape が変わりました。")
    for name, parameter in global_model.named_parameters():
        if torch.equal(parameter, before[name]):
            raise RuntimeError(f"Global の 1 步で更新されていない parameter があります: {name}")
    outside = {index: core.detach().clone() for index, core in enumerate(local_model.fc1.cores) if index not in {0, 1}}
    dense = {name: parameter.detach().clone() for name, parameter in local_model.named_parameters() if not name.startswith("fc1.cores.")}
    candidate = copy.deepcopy(local_model)
    local_result = update_tt_linear_two_core_pair(
        local_model, candidate, source, layer_name=TARGET_LAYER, pair_index=0, direction="right",
        center=0, rank_cap=RANK_CAP, tt_pair_learning_rate=0.3, dense_parameter_learning_rate=0.3,
        diagnostic_images=bundle["images"][:DIAGNOSTIC_BATCH_SIZE], diagnostic_labels=bundle["labels"][:DIAGNOSTIC_BATCH_SIZE],
        device=device, local_step=0, batch_size=GRADIENT_BATCH_SIZE,
    )
    if local_result.retained_rank > RANK_CAP:
        raise RuntimeError("Local の retained rank が上限を超えています。")
    for index, core in enumerate(local_model.fc1.cores):
        if index not in {0, 1} and not torch.equal(core, outside[index]):
            raise RuntimeError("Local の対象外 core が 1 步のあいだに変わりました。")
    if all(torch.equal(parameter, dense[name]) for name, parameter in local_model.named_parameters() if name in dense):
        raise RuntimeError("Local の joint 更新で通常 parameter が変わっていません。")
    same = initial_validation_selection(1.0, 0)
    same = update_validation_selection(same, 1.0, EVALUATION_INTERVAL, min_delta=MIN_DELTA, patience=PATIENCE)
    if same.early_stop_triggered or same.strict_best_step != 0:
        raise RuntimeError("同値 loss の扱いか、初期 best の扱いが selection API と違います。")


def run_smoke():
    if experiment_dirs(True)[0] == experiment_dirs(False)[0]:
        raise RuntimeError("smoke と main の保存先が同じです。")
    device = require_cuda_device()
    bundle = load_split()
    smoke_nonfinite()
    smoke_one_step(bundle, device)
    for method in METHODS:
        first = train_condition(method, 0.3, bundle, device, smoke=True)
        second = train_condition(method, 0.3, bundle, device, smoke=True)
        if not second["skipped"]:
            raise RuntimeError("完了した smoke を再学習しました。")
        if first["directory"] == condition_directory(False, method, 0.3):
            raise RuntimeError("smoke の state を main の条件ディレクトリへ書きました。")
    print("smoke は完了しました。main の学習には使っていません。")


## 図

横軸は、何を数えているかを名前にする。全コア同時GDの更新回数と DMRG-like（2コア局所更新）の往復回数は、同じ `full-dataset gradient evaluations` に揃えてから描く。学習時間の図は、完了した parameter 更新の累積秒であり、validation 評価、失敗試行、保存の秒は含めない。同じ勾配評価回数を同じ計算費用とは呼ばない。loss と accuracy は同じ軸に載せず、accuracy の単位は % である。peak memory は status が recorded の値だけを描く。Notebook 06 の記録値は測定範囲が未検証なので、同じ学習 peak としては描かない。欠損は 0 にしない。失敗条件の履歴は、失敗理由を付けて描き、正常完了や収束としては描かない。

図のタイトルは、どの方式が良かったかを先に述べない。


In [23]:
def configure_matplotlib():
    available = {font.name for font in font_manager.fontManager.ttflist}
    for name in ("Yu Gothic", "Meiryo", "MS Gothic"):
        if name in available:
            plt.rcParams["font.family"] = name
            break
    plt.rcParams["axes.unicode_minus"] = False


def save_figure(figure, path):
    assert_writable(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    display(Image(filename=str(path)))


def method_label(method):
    return {
        "global_gd": "全コア同時GD",
        "local_sweep": "DMRG-like（2コア局所更新）",
    }[method]


def condition_axis_label(method, learning_rate):
    rate = f"学習率={float(learning_rate):.1f}"
    if method == GLOBAL_METHOD:
        return f"全コア同時GD\n{rate}"
    if method == LOCAL_METHOD:
        return f"DMRG-like\n（2コア局所更新）\n{rate}"
    raise ValueError(f"未知の方式です: {method}")


def _finite_rows(rows, y_key):
    kept = []
    for row in rows:
        value = row.get(y_key)
        if value in {None, ""}:
            continue
        number = float(value)
        if math.isfinite(number):
            kept.append(row)
    return kept


def plot_curves(history, selected, path, x_key, x_label, y_key, y_label, title):
    figure, axis = plt.subplots(figsize=(10.8, 6.6))
    styles = {0.1: (0, (1, 1)), 0.2: (0, (4, 2)), 0.3: "solid"}
    colors = {GLOBAL_METHOD: "#4c78a8", LOCAL_METHOD: "#e45756"}
    for method in METHODS:
        for rate in LEARNING_RATES:
            rows = [
                row for row in history
                if row["method"] == method and float(row["learning_rate"]) == rate
            ]
            rows = _finite_rows(rows, y_key)
            rows = sorted(rows, key=lambda row: float(row[x_key]))
            if not rows:
                continue
            stop = rows[-1].get("condition_stop_reason") or ""
            failed = stop in {"non_finite", "out_of_memory"}
            chosen = (not failed) and selected.get(method) == rate
            label = f"{method_label(method)} 学習率={rate:.1f}"
            if chosen:
                label += "（選択）"
            elif failed:
                label += f"（失敗: {stop}）"
            axis.plot(
                [float(row[x_key]) for row in rows],
                [float(row[y_key]) * (100.0 if "accuracy" in y_key else 1.0) for row in rows],
                color=colors[method], linestyle=styles[rate], linewidth=2.6 if chosen else 1.2,
                label=label,
            )
        axis.set_xlabel(x_label)
    axis.set_ylabel(y_label)
    axis.set_title(title)
    legend_loc = "lower right" if "accuracy" in y_key else "upper right"
    axis.legend(loc=legend_loc, frameon=True, framealpha=0.92, fontsize=8, borderaxespad=0.35)
    figure.tight_layout()
    save_figure(figure, path)


def plot_best_final(summary, path):
    figure, axes = plt.subplots(2, 2, figsize=(12.4, 9.0))
    labels = [condition_axis_label(row["method"], row["learning_rate"]) for row in summary]
    positions = list(range(len(summary)))
    series = (
        (axes[0][0], "best_validation_loss", "final_validation_loss", "validation loss", "Validation loss"),
        (axes[0][1], "best_validation_accuracy", "final_validation_accuracy", "validation accuracy (%)", "Validation accuracy"),
        (axes[1][0], "best_gradient_evaluations", "final_gradient_evaluations", "勾配評価回数", "勾配評価回数"),
        (axes[1][1], "best_train_seconds", "final_train_seconds", "累積学習時間 (seconds)", "累積学習時間"),
    )
    for axis, best_key, final_key, y_label, subtitle in series:
        width = 0.36
        best_values = []
        final_values = []
        for row in summary:
            scale = 100.0 if "accuracy" in best_key else 1.0
            best_values.append(float(row[best_key]) * scale)
            final_values.append(float(row[final_key]) * scale)
        axis.bar([item - width / 2 for item in positions], best_values, width=width, label="最良時点", color="#4c78a8")
        axis.bar([item + width / 2 for item in positions], final_values, width=width, label="終了時点", color="#f2cf5b")
        axis.set_xticks(positions, labels)
        axis.set_ylabel(y_label)
        axis.set_title(subtitle)
    axes[0][1].legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=False)
    figure.suptitle("05｜全コア同時GDとDMRG-likeの最良時点・終了時点")
    figure.text(0.5, 0.01, "最良時点は、早期停止用の検証損失が最小だった時点。", ha="center", fontsize=9)
    figure.tight_layout(rect=(0, 0.06, 0.84, 0.94))
    save_figure(figure, path)


def plot_cost(summary, path):
    figure, axes = plt.subplots(1, 2, figsize=(22.5, 7.2))
    drawn = [row for row in summary if row.get("final_train_seconds") not in {None, ""}]
    labels = [condition_axis_label(row["method"], row["learning_rate"]) for row in drawn]
    positions = list(range(len(drawn)))
    seconds = [float(row["final_train_seconds"]) for row in drawn]
    axes[0].bar(positions, seconds, color="#4c78a8")
    axes[0].set_xticks(positions, labels)
    axes[0].tick_params(axis="x", labelsize=8)
    axes[0].set_ylabel("累積学習時間 (seconds)")
    axes[0].set_title("累積学習時間")
    memory = []
    memory_positions = []
    memory_labels = []
    for position, row in enumerate(summary):
        if row.get("peak_gpu_memory_status") != "recorded":
            continue
        if row.get("peak_gpu_memory_bytes") in {None, ""}:
            continue
        memory.append(float(row["peak_gpu_memory_bytes"]) / (1024 ** 3))
        memory_positions.append(position)
        memory_labels.append(condition_axis_label(row["method"], row["learning_rate"]))
    axes[1].bar(memory_positions, memory, color="#e45756")
    axes[1].set_xticks(memory_positions, memory_labels)
    axes[1].tick_params(axis="x", labelsize=8)
    axes[1].set_ylabel("Peak allocated GPU memory (GiB)")
    axes[1].set_title("Peak allocated GPU memory")
    figure.suptitle("06｜学習率別の学習費用とGPUメモリ")
    figure.text(
        0.5, 0.01,
        "全条件の完了した更新時間。失敗試行の秒は含めない。peak は recorded だけ。測定範囲が違う記録値は描かない。",
        ha="center", fontsize=9,
    )
    figure.tight_layout(rect=(0, 0.12, 1, 0.90))
    save_figure(figure, path)


def render_figures(history, selected_rows, all_rows, selected, results_dir):
    configure_matplotlib()
    figures = results_dir / "figures"
    loss_label = "validation loss"
    accuracy_label = "validation accuracy (%)"
    count_label = "全データ勾配評価回数"
    time_label = "累積学習時間 (seconds)"
    plot_curves(history, selected, figures / "01_gradient_evaluations_vs_validation_loss.png", "full_dataset_gradient_evaluations", count_label, "validation_loss", loss_label, "01｜全データ勾配評価回数とvalidation loss")
    plot_curves(history, selected, figures / "02_gradient_evaluations_vs_validation_accuracy.png", "full_dataset_gradient_evaluations", count_label, "validation_accuracy", accuracy_label, "02｜全データ勾配評価回数とvalidation accuracy")
    plot_curves(history, selected, figures / "03_train_seconds_vs_validation_loss.png", "cumulative_train_seconds", time_label, "validation_loss", loss_label, "03｜累積学習時間とvalidation loss")
    plot_curves(history, selected, figures / "04_train_seconds_vs_validation_accuracy.png", "cumulative_train_seconds", time_label, "validation_accuracy", accuracy_label, "04｜累積学習時間とvalidation accuracy")
    plot_best_final(selected_rows, figures / "05_best_final_comparison.png")
    plot_cost(all_rows, figures / "06_time_and_peak_memory.png")


## 実行

`RUN_SMOKE` だけを真にすると smoke を行い、main のディレクトリは使わない。`RUN_MAIN_EXPERIMENT` だけを真にすると、監査、再利用できる条件の再集計、足りない条件の学習、学習率の選択、rank-selection validation、図の保存まで進む。両方を真にはしない。`REGENERATE_DISPLAY_ARTIFACTS` だけを真にすると、保存済みの表から図だけを描き、学習はしない。

結果を見るまでは、次のどれに当たるかを埋めない。

- DMRG-like（2コア局所更新）が全コア同時GDより良かった。その場合でも、2コア更新、方向付き SVD、正準中心の移動、左右巡回を含む方式全体が良好だった、と書く。
- 同程度だった。その場合は「試した設定・予算・単一seedでは、DMRG-like（2コア局所更新）の追加利得を確認できなかった。」までに留める。
- 全コア同時GDが良かった。
- 一方が最大予算の時点でも改善中だった。
- 事前に固定した学習率 `(0.1, 0.2, 0.3)` の範囲に結論が限られる。
- validation 性能の順位と、実学習時間の順位が違う。

同じ勾配評価回数を同じ計算費用とは呼ばない。過去の差が full-batch 化だけによる、とも書かない。rank-selection validation の数値は、独立した最終試験の結果としては書かない。DMRG-like（2コア局所更新）の wall-clock time、total evaluation time、end-to-end の総費用は、各往復の開始時と終了時に行う評価の境界重複を含む。性能対学習時間の図は、その重複を含まない `cumulative_train_seconds` である。


In [24]:
def _summary_number(value):
    if value in {None, ""}:
        return None
    number = float(value)
    if not math.isfinite(number):
        return None
    return number


def completed_history(smoke):
    rows = []
    summaries = []
    rates = SMOKE_LEARNING_RATES if smoke else LEARNING_RATES
    for method in METHODS:
        for rate in rates:
            directory = condition_directory(smoke, method, rate)
            active = active_condition(directory, run_identifier(smoke))
            if active is None or active.manifest["stop_reason"] not in FINISHED_STOPS:
                raise RuntimeError(f"終了済みの条件がありません: {directory}")
            history = [coerce_row(row) for row in read_table(active.path / "history.csv")]
            state = json.loads((active.path / "run_state.json").read_text(encoding="utf-8"))
            config = json.loads((active.path / "config.json").read_text(encoding="utf-8"))
            if int(state["full_dataset_gradient_evaluations"]) != int(history[-1]["full_dataset_gradient_evaluations"]):
                raise RuntimeError("保存した更新数が、完了履歴の最終行と一致しません。")
            stop_reason = active.manifest["stop_reason"]
            for row in history:
                stamped = dict(row)
                stamped["condition_stop_reason"] = stop_reason
                rows.append(stamped)
            best = min(history, key=lambda row: (float(row["validation_loss"]), int(row["full_dataset_gradient_evaluations"])))
            final = history[-1]
            attempt = state.get("failed_attempt") or None
            if attempt is None and state.get("timing_scope") == TIMING_SCOPE:
                failed_train, failed_eval, failed_updates, rolled_back = 0.0, 0.0, 0, None
            elif attempt is None:
                failed_train, failed_eval, failed_updates, rolled_back = None, None, None, None
            else:
                failed_train = attempt.get("train_seconds")
                failed_eval = attempt.get("eval_seconds")
                failed_updates = attempt.get("attempted_updates")
                rolled_back = attempt.get("rolled_back_to_gradient_evaluations")
            summaries.append({
                "method": method,
                "learning_rate": rate,
                "stop_reason": stop_reason,
                "stop_detail": state.get("stop_detail"),
                "selection_status": "comparable" if stop_reason in COMPARISON_STOPS else "not_selectable",
                "best_validation_loss": best["validation_loss"],
                "final_validation_loss": final["validation_loss"],
                "best_validation_accuracy": best["validation_accuracy"],
                "final_validation_accuracy": final["validation_accuracy"],
                "best_gradient_evaluations": best["full_dataset_gradient_evaluations"],
                "final_gradient_evaluations": final["full_dataset_gradient_evaluations"],
                "best_train_seconds": best["cumulative_train_seconds"],
                "final_train_seconds": final["cumulative_train_seconds"],
                "seconds_train": state.get("seconds_train"),
                "seconds_eval": state.get("seconds_eval"),
                "failed_attempt_train_seconds": failed_train,
                "failed_attempt_eval_seconds": failed_eval,
                "failed_attempt_updates": failed_updates,
                "rolled_back_to_gradient_evaluations": rolled_back,
                "peak_gpu_memory_bytes": state.get("peak_gpu_memory_bytes"),
                "peak_gpu_memory_status": state.get("peak_gpu_memory_status"),
                "peak_memory_scope": state.get("peak_memory_scope"),
                "timing_scope": state.get("timing_scope"),
                "training_run_id": config.get("run_id"),
                "reuse_decision": config.get("reuse_decision"),
                "source_notebook": config.get("source_notebook"),
                "source_run_id": config.get("source_run_id"),
                "source_checkpoint": config.get("source_checkpoint"),
                "reaggregated": config.get("reaggregated"),
                "directory": str(active.path),
            })
    return rows, summaries


def select_learning_rates(summaries):
    selected = {}
    notes = {}
    for method in METHODS:
        present = [row for row in summaries if row["method"] == method]
        if len(present) != len(LEARNING_RATES):
            raise RuntimeError(f"{method} の学習率条件が揃っていません。")
        candidates = [row for row in present if row["stop_reason"] in COMPARISON_STOPS]
        if not candidates:
            selected[method] = None
            notes[method] = "選択可能条件なし"
            continue
        chosen = sorted(
            candidates,
            key=lambda row: (
                float(row["best_validation_loss"]),
                int(row["best_gradient_evaluations"]),
                float(row["best_train_seconds"]),
            ),
        )[0]
        selected[method] = float(chosen["learning_rate"])
        notes[method] = "selected"
    return selected, notes


def best_metrics_match(actual_loss, actual_accuracy, expected_loss, expected_accuracy):
    if not math.isfinite(float(actual_loss)) or not math.isfinite(float(actual_accuracy)):
        raise RuntimeError("再評価した validation 指標が有限ではありません。")
    loss_gap = abs(float(actual_loss) - float(expected_loss))
    if loss_gap > BEST_RECHECK_LOSS_ATOL:
        raise RuntimeError(
            f"再読込した best の validation loss が保存値と一致しません: "
            f"actual={actual_loss}, saved={expected_loss}, atol={BEST_RECHECK_LOSS_ATOL}"
        )
    accuracy_gap = abs(float(actual_accuracy) - float(expected_accuracy))
    if accuracy_gap > BEST_RECHECK_ACCURACY_ATOL:
        raise RuntimeError(
            f"再読込した best の validation accuracy が保存値と一致しません: "
            f"actual={actual_accuracy}, saved={expected_accuracy}, atol={BEST_RECHECK_ACCURACY_ATOL}"
        )


def assert_best_structure(model):
    if bond_ranks(model) != EXPECTED_ACTUAL_RANKS:
        raise RuntimeError("再読込した best の bond 次元が初期契約と一致しません。")
    for parameter in model.parameters():
        if not torch.isfinite(parameter).all():
            raise RuntimeError("再読込した best に非有限の parameter があります。")


def rank_selection_rows(bundle, device, summaries, selected):
    early_loader = make_loader(bundle["early_dataset"], VALIDATION_BATCH_SIZE)
    rank_loader = make_loader(bundle["rank_dataset"], VALIDATION_BATCH_SIZE)
    criterion = nn.CrossEntropyLoss(reduction="mean")
    rows = []
    note = "rank-selection validation は既に参照された開発用データであり、独立した最終評価ではない。学習率の選び直しには使わない。"
    for method, rate in selected.items():
        if rate is None:
            rows.append({
                "method": method,
                "learning_rate": None,
                "early_stopping_recheck_loss": None,
                "early_stopping_recheck_accuracy": None,
                "rank_selection_validation_loss": None,
                "rank_selection_validation_accuracy": None,
                "note": "選択可能条件なし。rank-selection validation は評価していない。",
            })
            continue
        summary = next(row for row in summaries if row["method"] == method and float(row["learning_rate"]) == rate)
        model = build_stage_c_model()
        load_tt_linear_sweep_checkpoint(Path(summary["directory"]) / "best_model.pt", model)
        model.to(device)
        recheck_loss, recheck_accuracy = evaluate(model, early_loader, criterion, device)
        best_metrics_match(recheck_loss, recheck_accuracy, summary["best_validation_loss"], summary["best_validation_accuracy"])
        assert_best_structure(model)
        loss, accuracy = evaluate(model, rank_loader, criterion, device)
        rows.append({
            "method": method,
            "learning_rate": rate,
            "early_stopping_recheck_loss": float(recheck_loss),
            "early_stopping_recheck_accuracy": float(recheck_accuracy),
            "rank_selection_validation_loss": float(loss),
            "rank_selection_validation_accuracy": float(accuracy),
            "note": note,
        })
    return rows


def search_cost_record(summaries, selected, notes):
    def total(key):
        values = []
        unknown = []
        for row in summaries:
            number = _summary_number(row.get(key))
            label = f"{row['method']}:{row['learning_rate']}"
            if number is None:
                unknown.append(label)
            else:
                values.append(number)
        return {
            "recorded_sum": None if unknown or not values else sum(values),
            "complete": not unknown,
            "unknown_conditions": unknown,
        }

    return {
        "completed_train_seconds": total("seconds_train"),
        "completed_eval_seconds": total("seconds_eval"),
        "failed_attempt_train_seconds": total("failed_attempt_train_seconds"),
        "failed_attempt_eval_seconds": total("failed_attempt_eval_seconds"),
        "selected_learning_rates": selected,
        "selection_notes": notes,
        "cost_note": (
            "完了した更新秒、評価秒、失敗試行の秒は別項目。"
            "同じ full-dataset gradient evaluation 数は同じ計算費用ではない。"
            "保存の overhead はこれらの秒に含めない。"
            "wall-clock time、total evaluation time、end-to-end の総費用には、Local の境界評価の重複を含む。"
        ),
    }



def build_results_manifest(summaries, smoke):
    settings = training_settings(smoke)
    conditions = []
    for row in summaries:
        # summaries の directory は世代パス（best_model 読込用）。pointer は条件ディレクトリ側。
        directory = condition_directory(smoke, row["method"], float(row["learning_rate"]))
        active = active_condition(directory, row["training_run_id"])
        if active is None:
            raise RuntimeError(f"condition の世代がありません: {directory}")
        conditions.append({
            "method": row["method"],
            "learning_rate": float(row["learning_rate"]),
            "generation": int(active.generation),
            "condition_run_id": active.manifest["run_id"],
        })
    return {
        "training_run_id": training_run_identifier(smoke),
        "training_contract_version": TRAINING_CONTRACT_VERSION,
        "initial_checkpoint_sha256": STAGE_C_SHA256,
        "split_index_sha256": SPLIT_INDEX_SHA256,
        "implementation_sha256": settings["implementation_sha256"],
        "learning_rates": list(settings["learning_rates"]),
        "evaluation_interval": settings["evaluation_interval"],
        "max_gradient_evaluations": settings["max_gradient_evaluations"],
        "min_delta": MIN_DELTA,
        "patience": PATIENCE,
        "conditions": conditions,
    }


def require_saved_experiment(results_dir):
    """図を描く前に、保存済みの学習契約と集計が同じ実験であることを確認する。

    図の再生成は保存済み CSV から描く。学習設定・データ・contract version が同じなら、
    src 実装 hash だけのずれは警告に留め、再描画を許す。
    """
    manifest_path = results_dir / "results_manifest.json"
    summary_path = results_dir / "condition_summary.csv"
    if not manifest_path.is_file() or not summary_path.is_file():
        raise RuntimeError("results_manifest.json または集計 CSV がありません。図は上書きしません。")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    summaries = read_table(summary_path)
    settings = training_settings(False)
    expected_id = training_run_identifier(False)
    if manifest.get("training_contract_version") != TRAINING_CONTRACT_VERSION:
        raise RuntimeError("training contract version が違います。図は上書きしません。")
    for key, current in (
        ("learning_rates", list(settings["learning_rates"])),
        ("evaluation_interval", settings["evaluation_interval"]),
        ("max_gradient_evaluations", settings["max_gradient_evaluations"]),
        ("min_delta", MIN_DELTA),
        ("patience", PATIENCE),
        ("initial_checkpoint_sha256", STAGE_C_SHA256),
        ("split_index_sha256", SPLIT_INDEX_SHA256),
    ):
        if manifest.get(key) != current:
            raise RuntimeError(f"保存済み manifest の {key} が現在の学習設定と違います。図は上書きしません。")
    if manifest.get("training_run_id") != expected_id:
        probe = dict(settings)
        probe["implementation_sha256"] = manifest.get("implementation_sha256")
        probed = hashlib.sha256(
            json.dumps(probe, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
        ).hexdigest()
        if probed == manifest.get("training_run_id"):
            print(
                "警告: src 実装 hash が保存時と違います。"
                "保存済み履歴から図だけ再生成します。"
                f" saved={manifest.get('implementation_sha256')} current={settings['implementation_sha256']}"
            )
        else:
            raise RuntimeError("training run ID が現在の学習設定と違います。図は上書きしません。")
    csv_ids = {row.get("training_run_id") for row in summaries}
    if csv_ids != {manifest["training_run_id"]}:
        raise RuntimeError("集計 CSV の training run ID が manifest と違います。図は上書きしません。")
    expected_pairs = {(method, float(rate)) for method in METHODS for rate in LEARNING_RATES}
    csv_pairs = {(row["method"], float(row["learning_rate"])) for row in summaries}
    manifest_pairs = {(item["method"], float(item["learning_rate"])) for item in manifest.get("conditions", [])}
    if len(manifest.get("conditions", [])) != len(expected_pairs) or csv_pairs != expected_pairs or manifest_pairs != expected_pairs:
        raise RuntimeError("condition の method と learning rate が想定と違います。図は上書きしません。")
    for item in manifest["conditions"]:
        directory = condition_directory(False, item["method"], float(item["learning_rate"]))
        generation_dir = directory / f"generation_{int(item['generation']):06d}"
        generation_manifest = generation_dir / "manifest.json"
        if not generation_manifest.is_file():
            raise RuntimeError(f"参照先 generation がありません: {generation_dir}。図は上書きしません。")
        saved = json.loads(generation_manifest.read_text(encoding="utf-8"))
        if saved.get("run_id") != item["condition_run_id"]:
            raise RuntimeError("generation の condition run ID が一致しません。図は上書きしません。")
    return manifest


def export_results(history, summaries, selected, notes, audit, rank_rows, results_dir):
    results_dir.mkdir(parents=True, exist_ok=True)
    atomic_write_csv(results_dir / "history.csv", history, HISTORY_FIELDS)
    summary_fields = [key for key in summaries[0] if key != "directory"]
    atomic_write_csv(results_dir / "condition_summary.csv", [{key: row[key] for key in summary_fields} for row in summaries], summary_fields)
    audit_fields = ["source_notebook", "source_run_id", "source_checkpoint", "reuse_decision", "reaggregated", "reason", "method", "learning_rate"]
    atomic_write_csv(results_dir / "reuse_audit.csv", audit, audit_fields)
    chosen = [
        row for row in summaries
        if selected.get(row["method"]) is not None and selected.get(row["method"]) == float(row["learning_rate"])
    ]
    atomic_write_csv(results_dir / "selected_comparison.csv", [{key: row[key] for key in summary_fields} for row in chosen], summary_fields)
    atomic_write_csv(
        results_dir / "rank_selection.csv", rank_rows,
        [
            "method", "learning_rate", "early_stopping_recheck_loss", "early_stopping_recheck_accuracy",
            "rank_selection_validation_loss", "rank_selection_validation_accuracy", "note",
        ],
    )
    atomic_write_json(results_dir / "search_cost.json", search_cost_record(summaries, selected, notes))
    atomic_write_json(results_dir / "results_manifest.json", build_results_manifest(summaries, smoke=False))
    render_figures(history, chosen, summaries, selected, results_dir)


def redraw_saved_figures():
    _models_dir, results_dir = experiment_dirs(False)
    require_saved_experiment(results_dir)
    history = [coerce_row(row) for row in read_table(results_dir / "history.csv")]
    summaries = read_table(results_dir / "condition_summary.csv")
    for row in summaries:
        for key in ("learning_rate", "best_validation_loss", "final_validation_loss", "best_validation_accuracy", "final_validation_accuracy", "best_train_seconds", "final_train_seconds"):
            row[key] = float(row[key])
        for key in ("best_gradient_evaluations", "final_gradient_evaluations"):
            row[key] = int(float(row[key]))
    selected, _notes = select_learning_rates(summaries)
    chosen = [
        row for row in summaries
        if selected.get(row["method"]) is not None and selected.get(row["method"]) == float(row["learning_rate"])
    ]
    render_figures(history, chosen, summaries, selected, results_dir)


def run_main():
    device = require_cuda_device()
    bundle = load_split()
    _models_dir, results_dir = experiment_dirs(False)
    audit, decision = audit_rows()
    run_id = run_identifier(False)
    if decision == "reuse_reaggregate":
        source_run = next(row["source_run_id"] for row in audit if row["reuse_decision"] == "reuse_reaggregate")
        reaggregate_notebook06(run_id, source_run)
    elif decision != "new_run":
        raise RuntimeError(f"未知の再利用判定です: {decision}")
    for method in METHODS:
        for rate in LEARNING_RATES:
            if method == LOCAL_METHOD and rate == 0.3 and decision == "reuse_reaggregate":
                continue
            train_condition(method, rate, bundle, device, smoke=False)
    history, summaries = completed_history(False)
    selected, notes = select_learning_rates(summaries)
    rank_rows = rank_selection_rows(bundle, device, summaries, selected)
    export_results(history, summaries, selected, notes, audit, rank_rows, results_dir)


def main():
    if RUN_SMOKE and RUN_MAIN_EXPERIMENT:
        raise RuntimeError("smoke と main は同時に実行しません。")
    if not (RUN_SMOKE or RUN_MAIN_EXPERIMENT or REGENERATE_DISPLAY_ARTIFACTS):
        print("実行スイッチはすべて False です。学習も図の再生成も行いません。")
        return
    if REGENERATE_DISPLAY_ARTIFACTS and not RUN_SMOKE and not RUN_MAIN_EXPERIMENT:
        redraw_saved_figures()
        return
    if RUN_SMOKE:
        run_smoke()
        return
    run_main()


main()


RuntimeError: 保存済み世代の run_id が現在の設定と違います。上書きせず停止します: D:\dev\nn-compression-svd-dmrg\models\40_dmrg\10_fashion_mnist_mlp\11_full_batch_global_vs_local_update_control\global_gd\lr0p1 saved=58d5b7e1074b6cb1c7134dd6b8bc2f875753189b331ff95352bd986cf73945e3 current=3de6e7b16851756234cbbc9822a3064911925d42f553f76bad33bada4fe5ff5a。実装や学習設定が変わった場合は、その条件ディレクトリを削除してから再実行してください。